# California Housing Dataset

In [69]:
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

random_seed = 42
torch.manual_seed(random_seed)

In [70]:
# Load the dataset
X, y = fetch_california_housing(return_X_y=True)
X.shape, y.shape

((20640, 8), (20640,))

In [71]:
# Split the dataset, 20% for testing, with random seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((16512, 8), (4128, 8), (16512,), (4128,))

In [72]:
# Standardize the data using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

last_test = X_test[-1]

X_train.shape, X_test.shape, np.min(last_test), np.max(last_test)

((16512, 8),
 (4128, 8),
 np.float64(-0.9211376319711084),
 np.float64(0.6044549279675977))

In [73]:
# Convert the data to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float().reshape(-1, 1)
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float().reshape(-1, 1)

X_train_tensor.shape, y_train_tensor.shape, X_test_tensor.shape, y_test_tensor.shape

(torch.Size([16512, 8]),
 torch.Size([16512, 1]),
 torch.Size([4128, 8]),
 torch.Size([4128, 1]))

In [74]:
# Defining the model, RegresionANN
class RegressionANN(nn.Module):
    def __init__(self, input_size=8, hidden_size=16, output_size=1):
        super(RegressionANN, self).__init__()
        self.hidden_layer = nn.Linear(input_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.hidden_layer(x)
        x = F.relu(x)
        x = self.output_layer(x)
        return x

In [75]:
# Instantiate the model
model = RegressionANN(hidden_size=16) 
w1, b1, w2, b2 = list(model.parameters())
b2

Parameter containing:
tensor([0.2272], requires_grad=True)

In [76]:
# Initialize loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [77]:
# Running a forward pass
y_pred = model(X_train_tensor)
loss = criterion(y_pred, y_train_tensor)
loss.item()

4.202902793884277

In [78]:
# Training the model for 100 epochs
epochs = 100
for epoch in range(1,epochs+1):
    optimizer.zero_grad() # Zero the gradients
    y_pred = model(X_train_tensor) # Forward pass
    loss = criterion(y_pred, y_train_tensor) # Compute the loss
    loss.backward() # Backward pass
    optimizer.step() # Update the weights
    if epoch % 10 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.detach().item()}')

Epoch: 10, Loss: 2.333400011062622
Epoch: 20, Loss: 1.119511365890503
Epoch: 30, Loss: 0.8087179660797119
Epoch: 40, Loss: 0.7286894917488098
Epoch: 50, Loss: 0.6494651436805725
Epoch: 60, Loss: 0.6001060009002686
Epoch: 70, Loss: 0.5567218661308289
Epoch: 80, Loss: 0.5240655541419983
Epoch: 90, Loss: 0.497019499540329
Epoch: 100, Loss: 0.4749816954135895


In [79]:
# Changing hidden layer size to 64
model = RegressionANN(hidden_size=64)
w1, b1, w2, b2 = list(model.parameters())

In [80]:
# Training the model for 100 epochs
epochs = 100
optimizer = optim.Adam(model.parameters(), lr=0.01) # Reinitialize the optimizer
for epoch in range(1,epochs+1):
    optimizer.zero_grad() # Zero the gradients
    y_pred = model(X_train_tensor) # Forward pass
    loss = criterion(y_pred, y_train_tensor) # Compute the loss
    loss.backward() # Backward pass
    optimizer.step() # Update the weights
    if epoch % 10 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.detach().item()}')

Epoch: 10, Loss: 1.0591105222702026
Epoch: 20, Loss: 0.9692996144294739
Epoch: 30, Loss: 0.682636022567749
Epoch: 40, Loss: 0.5752586126327515
Epoch: 50, Loss: 0.5252892374992371
Epoch: 60, Loss: 0.4831806719303131
Epoch: 70, Loss: 0.4564894437789917
Epoch: 80, Loss: 0.4391113221645355
Epoch: 90, Loss: 0.42621374130249023
Epoch: 100, Loss: 0.41575032472610474
